# Experiment: Case-001 Mitapivat Seven Domain Owner Action Map

Objective:
- Validate the seven-domain owner action map before clinician outreach.
- Confirm each domain has an owner, public-safe label, and missing-data blocker.

Success criteria:
- seven required domains are present;
- every domain has a primary owner and owner question;
- public labels stay label-only and do not contain raw clinical instructions.


In [ ]:
# Setup: imports and source path
from __future__ import annotations

import json
from pathlib import Path

DATA_PATH = Path(
    "data/regulatory/mitapivat/2026-05-20-mitapivat-seven-domain-owner-action-map.json"
)
gate = json.loads(DATA_PATH.read_text(encoding="utf-8"))
gate["gate_label"]

## Gate Contract

The map turns packet readiness into owner-specific private record questions.
It does not decide treatment, dosing, access, import, travel, or trial
screening.


In [ ]:
required_domains = {
    "diagnosis_genotype_context",
    "age_weight_context",
    "transfusion_burden_context",
    "iron_organ_context",
    "liver_safety_context",
    "medication_interaction_context",
    "local_access_context",
}

required_next_states = {
    "domain_missing_private_record_request",
    "domain_label_ready_private",
    "domain_release_blocked_public_label_only",
    "all_domains_ready_for_clinician_question",
}

blocked_public_terms = {
    "dose",
    "dosing",
    "treatment change",
    "trial screening",
    "import",
    "travel",
    "doctor message",
    "screenshot",
    "contact detail",
}


def has_blocked_public_term(text: str) -> bool:
    """Return True when public labels contain blocked instruction terms."""
    normalized = text.lower()
    return any(term in normalized for term in blocked_public_terms)


matrix = gate["domain_owner_matrix"]
assert gate["gate_label"] == "mitapivat_seven_domain_owner_action_map_ready"
assert gate["upstream_gate"] == "mitapivat_minimum_packet_readiness_ready"
assert {item["domain_label"] for item in matrix} == required_domains
assert set(gate["allowed_next_states"]) == required_next_states
assert len(gate["source_checks"]) >= 6

for item in matrix:
    assert item["primary_owner"]
    assert item["minimum_private_evidence"]
    assert item["owner_question"]
    assert item["public_safe_label"]
    assert item["blocks_if_missing"]
    assert not has_blocked_public_term(item["public_safe_label"])

decision = {
    "gate_label": gate["gate_label"],
    "domains_checked": len(matrix),
    "next_states_checked": len(gate["allowed_next_states"]),
    "public_action": "record_owner_categories_and_labels_only",
}
decision

## Results

- The map covers all seven May 19 packet domains.
- Each domain has a private evidence target and owner category.
- Public outputs stay limited to labels, owner categories, source URLs, and
  safe questions.


## Next steps

- Keep raw record retrieval outside Git.
- Resolve missing domains with the correct clinical owner first.
- Ask the May 17 clinician question only after every domain is privately
  label-ready and release-checked.
